# w9_student_scan.ipynb — 学生视图扩展（W 扫描，Fig4 的镜像）

User (2026-07-22): 教师扩展（锚点 512→4096）已在 Fig4；学生扩展反向——**放大学生视图 W 反而伤检索**（固定分割 g2048: W16 .730 → W48 .672 / W64 .686，tag 微升）。此本把它做成五折、同预算的干净折线。i2ce **@512**（教师水平线固定，横向比 W 受控）× W∈{16,32,48,64} × 5折 × 2000ep。W=16@512 复用卷上现成塔；只跑 32/48/64。cvsel 选点，test_tag 报告。AUTO-STOPS。


In [ ]:
# constants
import os
REPO = "/workspace/stable-query-latent"
URL = "https://github.com/Nice9Tian/stable-query-latent.git"
DATA_SRC = "/workspace/fusion_cache_w9"
DATA_RAM = "/dev/shm/fusion_cache_w9"
OUT_DIR = "/workspace/w9_cv_out"
os.makedirs(OUT_DIR, exist_ok=True)
print("student-scan @512: i2ce x W{16,32,48,64} x 5 folds")


In [ ]:
# FORCE-sync repo to origin/main.
import os, importlib.util
if not os.path.isdir(os.path.join(REPO, ".git")):
    !git clone {URL} {REPO}
%cd {REPO}
!git remote set-url origin {URL}
!git fetch origin main
!git reset --hard origin/main
!git rev-parse --short HEAD
for pkg in ("sklearn", "scipy"):
    if importlib.util.find_spec(pkg) is None:
        !pip -q install scikit-learn scipy
        break
import sys
if REPO not in sys.path:
    sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, "Pod"))
import w9_jobs as J
print("machinery loaded")


In [ ]:
# Stage the corpus into RAM (same file set as w9_a100.ipynb).
import shutil
from pathlib import Path
REQUIRED = ["games.npz", "wiki_eval.npz", "wscan_gal_rev.npz",
            "wscan_pool_rev.npy", "wscan_pool_rev_rid.npy", "wscan_pool_rev_len.npy",
            "ss_queries_rev.npz", "ss_queries_rev_S.npy",
            "wiki_clean_views.npz", "sp_raw_views.npz", "tag_labels.npz",
            "wiki_eval_split.json", "_tag_splitM.json"]
src = Path(DATA_SRC)
missing = [f for f in REQUIRED if not (src / f).exists()]
assert not missing, f"missing in {DATA_SRC}: {missing}"
dst = Path(DATA_RAM)
dst.mkdir(parents=True, exist_ok=True)
for f in REQUIRED:
    s, d = src / f, dst / f
    if not d.exists() or d.stat().st_size != s.stat().st_size:
        print(f"staging {f} ({s.stat().st_size/1e9:.2f} GB) ...", flush=True)
        shutil.copyfile(s, d)
DATA_DIR = str(dst)
print("corpus in RAM:", DATA_DIR)

In [ ]:
# Student-view W scan: i2ce @512, W in {32,48,64}, five folds.
import os, subprocess, threading, time
from pathlib import Path

ARM, CAP, EPOCHS, N_FOLDS = "wcle_i2ce_icetf", 512, 2000, 5
FULL_POOL_PATH = f"{DATA_SRC}/full_pool_fp16.npy"   # mmap, not staged (name parity with W16@512 _fp)
WS = [32, 48, 64]                    # W=16@512 already on the volume
cdir = Path(OUT_DIR) / "claims"; cdir.mkdir(parents=True, exist_ok=True)
logd = Path(OUT_DIR) / "logs"; logd.mkdir(parents=True, exist_ok=True)
gpus = J.detect_gpus()

todo = []
for W in WS:
    for k in range(N_FOLDS):
        nm = f"w9cv_{ARM}_fold{k}_w{W}_fp"       # match W16@512 baseline (_fp)
        if (Path(OUT_DIR) / f"tower_{nm}_ep{EPOCHS}.npz").exists():
            print(f"[skip] {nm} done"); continue
        todo.append((W, k, nm))
print(f"{len(todo)} towers to train ({len(WS)} W x {N_FOLDS} folds, minus done)")

stop_evt = threading.Event()
threading.Thread(target=J._monitor, args=([logd], stop_evt), daemon=True).start()
fails, retried = [], set()
lock = threading.Lock()
pending = list(todo)

def run_one(gpu):
    while True:
        with lock:
            if not pending: return
            W, k, nm = pending.pop(0)
        if not J.try_claim(cdir, nm):
            print(f"[claim] {nm} held elsewhere -- skip", flush=True); continue
        cmd = ["python", "-u", J.CV_WORKER, "--data-dir", DATA_DIR, "--out-dir",
               OUT_DIR, "--repo", REPO, "--arm", ARM, "--fold", str(k),
               "--n-folds", str(N_FOLDS), "--anchor-cap", str(CAP),
               "--view-w", str(W), "--epochs", str(EPOCHS), "--ckpt-every", "50",
               "--full-pool", "--full-pool-path", FULL_POOL_PATH,
               "--claim-file", str(cdir / f"{nm}.claim")]
        print(f"[gpu{gpu}] start {nm}", flush=True)
        t0 = time.time()
        with open(logd / f"{nm}.log", "w") as fh:
            rc = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT,
                                env=dict(os.environ, CUDA_VISIBLE_DEVICES=gpu,
                                         PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True"))
        ok = (Path(OUT_DIR) / f"tower_{nm}_ep{EPOCHS}.npz").exists()
        if rc.returncode != 0 or not ok:
            (cdir / f"{nm}.claim").unlink(missing_ok=True)
            with lock:
                if nm not in retried:
                    retried.add(nm); pending.append((W, k, nm))
                    print(f"[retry] {nm} re-queued once", flush=True)
                else:
                    fails.append(nm)
        print(f"[gpu{gpu}] {'ok' if ok else 'FAIL'} {nm} [{(time.time()-t0)/3600:.1f}h]",
              flush=True)

ths = [threading.Thread(target=run_one, args=(g,)) for g in gpus]
for t in ths: t.start()
for t in ths: t.join()
stop_evt.set()
print(f"done; {len(fails)} failed: {fails}")


In [ ]:
# Student-scan readout: W vs stripped hit@1 / test_tag (five-fold).
import json
import numpy as np
from pathlib import Path

ARM, N_FOLDS = "wcle_i2ce_icetf", 5
def rows(W):
    out = []
    for k in range(N_FOLDS):
        sfx = "" if W == 16 else f"_w{W}"
        p = Path(OUT_DIR) / f"zsbest_w9cv_{ARM}_fold{k}{sfx}_fp.json"
        if p.exists(): out.append(json.loads(p.read_text()))
    return out
print(f"{'W':>4} {'n':>3}  {'non h1':>12} {'non h5':>12} {'test_tag':>12}")
for W in (16, 32, 48, 64):
    r = rows(W)
    if not r: print(f"{W:>4}  (pending)"); continue
    M = lambda f: np.mean([x[f] for x in r]); S = lambda f: np.std([x[f] for x in r])
    print(f"{W:>4} {len(r):>3}  {M('nm_noname'):.3f}±{S('nm_noname'):.3f} "
          f"{M('h5_noname'):.3f}±{S('h5_noname'):.3f} {M('test_tag'):.3f}±{S('test_tag'):.3f}")


In [ ]:
# AUTO-STOP: stop THIS pod when the queue has finished (results live on the
# network volume; idle GPU time is pure waste). Uses the hardened ladder in
# VICReg_review/pod_selfstop.py. Set AUTO_STOP=False to keep the pod alive.
AUTO_STOP = True
if AUTO_STOP:
    import sys
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    from VICReg_review import pod_selfstop
    if fails:
        print(f"NOTE: {len(fails)} job(s) FAILED -- logs in {OUT_DIR}/logs; "
              "stopping anyway to avoid idle burn.")
    pod_id, api_key, ctl = pod_selfstop.preflight("")
    pod_selfstop.stop_pod(pod_id, api_key, ctl)
else:
    print("AUTO_STOP disabled -- remember to stop the pod yourself.")